# 导入依赖和定义文件

In [7]:

import json

import joblib
import numpy as np
import optuna
import pandas as pd
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split

In [8]:

# 加载特征数据
X = np.load("X.npy")
X_test = np.load("X_test.npy")
y = np.load("y.npy")

# 划分训练集与验证集

X_train, X_valid, y_train, y_valid = train_test_split(X, y, stratify=y, train_size=0.8, test_size=0.2, random_state=0)


# 寻找并保存最优参数

In [9]:

# 定义 LGBM 的目标函数
def lgbm_objective(trial):
    params = {
        'n_estimators': trial.suggest_categorical('n_estimators', [50, 100]),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 4, 6),
        'num_leaves': trial.suggest_int('num_leaves', 20, 30),
        'subsample': trial.suggest_float('subsample', 0.8, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.8, 1.0),
        'random_state': 0
    }
    model = LGBMClassifier(**params)
    model.fit(X_train, y_train)
    return model.score(X_valid, y_valid)


# 定义 CatBoost 的目标函数
def catboost_objective(trial):
    params = {
        'n_estimators': trial.suggest_categorical('n_estimators', [50, 100]),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'depth': trial.suggest_int('depth', 4, 6),
        'l2_leaf_reg': trial.suggest_int('l2_leaf_reg', 1, 3),
        'border_count': trial.suggest_categorical('border_count', [32, 64]),
        'verbose': False,
        'random_state': 0
    }
    model = CatBoostClassifier(**params)
    model.fit(X_train, y_train)
    return model.score(X_valid, y_valid)


# 创建 Optuna 研究对象并进行优化
lgbm_study = optuna.create_study(direction='maximize')
lgbm_study.optimize(lgbm_objective, n_trials=50)

catboost_study = optuna.create_study(direction='maximize')
catboost_study.optimize(catboost_objective, n_trials=50)

# 获取最优参数
lgbm_best_params = lgbm_study.best_params
lgbm_best_value = lgbm_study.best_value
catboost_best_params = catboost_study.best_params
catboost_best_value = catboost_study.best_value

[I 2025-04-26 19:37:21,497] A new study created in memory with name: no-name-d874f1b4-122e-40e5-979a-b6d6c98ebee9
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-26 19:37:21,536] Trial 0 finished with value: 0.7872340425531915 and parameters: {'n_estimators': 50, 'learning_rate': 0.047046738549755975, 'max_depth': 5, 'num_leaves': 30, 'subsample': 0.9024583790204905, 'colsample_bytree': 0.9479634091652078}. Best is trial 0 with value: 0.7872340425531915.
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-26 19:37:21,578] Trial 1 finished with value: 0.8004600345025877 and parameters: {'n_estimators': 100, 

[LightGBM] [Info] Number of positive: 3502, number of negative: 3452
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000296 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2460
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 38
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503595 -> initscore=0.014380
[LightGBM] [Info] Start training from score 0.014380
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -i

/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-26 19:37:21,730] Trial 4 finished with value: 0.772857964347326 and parameters: {'n_estimators': 100, 'learning_rate': 0.011338311462916945, 'max_depth': 6, 'num_leaves': 29, 'subsample': 0.9301634394483483, 'colsample_bytree': 0.9535298163964716}. Best is trial 1 with value: 0.8004600345025877.
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-26 19:37:21,775] Trial 5 finished with value: 0.7912593444508338 and parameters: {'n_estimators': 100, 'learning_rate': 0.0386426251467067, 'max_depth': 4, 'num_leaves': 27, 'subsample': 0.8930727290246757, 'colsample

[LightGBM] [Info] Number of positive: 3502, number of negative: 3452
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000311 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2460
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 38
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503595 -> initscore=0.014380
[LightGBM] [Info] Start training from score 0.014380
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -i

/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-26 19:37:21,982] Trial 9 finished with value: 0.7849338700402531 and parameters: {'n_estimators': 100, 'learning_rate': 0.016388430605102998, 'max_depth': 6, 'num_leaves': 25, 'subsample': 0.9409539373835643, 'colsample_bytree': 0.8065496899620936}. Best is trial 1 with value: 0.8004600345025877.
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-26 19:37:22,057] Trial 10 finished with value: 0.8010350776308223 and parameters: {'n_estimators': 100, 'learning_rate': 0.08852248278855344, 'max_depth': 4, 'num_leaves': 24, 'subsample': 0.9833005256684282, 'colsam

...

/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-26 19:37:24,124] Trial 46 finished with value: 0.8010350776308223 and parameters: {'n_estimators': 100, 'learning_rate': 0.06077312223327492, 'max_depth': 6, 'num_leaves': 22, 'subsample': 0.9899108977099712, 'colsample_bytree': 0.9230200163194958}. Best is trial 45 with value: 0.8062104657849338.
/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-26 19:37:24,190] Trial 47 finished with value: 0.7993099482461185 and parameters: {'n_estimators': 100, 'learning_rate': 0.08705393024475773, 'max_depth': 6, 'num_leaves': 21, 'subsample': 0.9715290772550562, 'colsa

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Number of positive: 3502, number of negative: 3452
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000589 seconds.
Yo

/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2025-04-26 19:37:24,317] Trial 49 finished with value: 0.7987349051178838 and parameters: {'n_estimators': 100, 'learning_rate': 0.09259554577296102, 'max_depth': 6, 'num_leaves': 20, 'subsample': 0.978810362357458, 'colsample_bytree': 0.864190704305353}. Best is trial 45 with value: 0.8062104657849338.
[I 2025-04-26 19:37:24,318] A new study created in memory with name: no-name-bbe40ae6-865c-4b12-b616-2bb8c8ad0606


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


[I 2025-04-26 19:37:24,562] Trial 0 finished with value: 0.7987349051178838 and parameters: {'n_estimators': 100, 'learning_rate': 0.0371003327548131, 'depth': 6, 'l2_leaf_reg': 1, 'border_count': 32}. Best is trial 0 with value: 0.7987349051178838.
[I 2025-04-26 19:37:24,747] Trial 1 finished with value: 0.7918343875790684 and parameters: {'n_estimators': 50, 'learning_rate': 0.07533939680150116, 'depth': 6, 'l2_leaf_reg': 1, 'border_count': 64}. Best is trial 0 with value: 0.7987349051178838.
[I 2025-04-26 19:37:25,157] Trial 2 finished with value: 0.7855089131684876 and parameters: {'n_estimators': 50, 'learning_rate': 0.02838261414584707, 'depth': 6, 'l2_leaf_reg': 1, 'border_count': 32}. Best is trial 0 with value: 0.7987349051178838.
[I 2025-04-26 19:37:25,469] Trial 3 finished with value: 0.777458309373203 and parameters: {'n_estimators': 50, 'learning_rate': 0.03356790021319769, 'depth': 4, 'l2_leaf_reg': 1, 'border_count': 64}. Best is trial 0 with value: 0.7987349051178838.
[

In [10]:
print("LGBM 最佳得分:", lgbm_best_value)
print("CatBoost 最佳得分:", catboost_best_value)
# 导出 LGBM 和 CatBoost 的最优参数到 JSON 文件
with open('lgbm_best_params.json', 'w') as f:
    json.dump(lgbm_best_params, f)

with open('catboost_best_params.json', 'w') as f:
    json.dump(catboost_best_params, f)

LGBM 最佳得分: 0.8062104657849338
CatBoost 最佳得分: 0.8067855089131685


# 训练并保存模型

In [11]:

# 导入最优参数
with open('lgbm_best_params.json', 'r') as f:
    lgbm_best_params = json.load(f)

with open('catboost_best_params.json', 'r') as f:
    catboost_best_params = json.load(f)

# 使用最优参数重新实例化模型并训练
lgbm_model = LGBMClassifier(**lgbm_best_params, random_state=0)
catboost_model = CatBoostClassifier(**catboost_best_params, verbose=False, random_state=0)

lgbm_model.fit(X_train, y_train)
catboost_model.fit(X_train, y_train)

joblib.dump(LGBMClassifier, 'lgbm_best_model.joblib')
joblib.dump(CatBoostClassifier, 'catboost_best_model.joblib')


[LightGBM] [Info] Number of positive: 3502, number of negative: 3452
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000492 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2460
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 38
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503595 -> initscore=0.014380
[LightGBM] [Info] Start training from score 0.014380
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

['catboost_best_model.joblib']

# 提交文件

In [12]:
# 加载模型
lgbm_loaded_model = joblib.load('lgbm_best_model.joblib')
catboost_loaded_model = joblib.load('catboost_best_model.joblib')

# 对测试数据进行预测，获取概率值
lgbm_prob = lgbm_model.predict_proba(X_test)[:, 1]
catboost_prob = catboost_model.predict_proba(X_test)[:, 1]

# 简单平均融合
ensemble_prob = (lgbm_prob + catboost_prob) / 2

# 根据阈值生成最终预测
threshold = 0.5
ensemble_pred = ensemble_prob > threshold

# 加载测试数据
test_data = pd.read_csv('test.csv')  # 确保文件路径正确

# 创建提交文件
submission = pd.DataFrame({
    'PassengerId': test_data['PassengerId'],  # 确保这里使用正确的列名
    'Transported': ensemble_pred
})

# 保存为 CSV 文件
submission.to_csv('submission.csv', index=False)

/home/gyh/Documents/文档/数据挖掘/spaceship-titanic/venv/lib64/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
